##### HOSPITAL AI POLICY ASSISTANT (RAG)
- imagine a hospital has thousands of policy documents.
- patients ask questions like:
1. Can I cancel my appointment?
2. Do you cover dental insurance?
3. What is the visiting time?
4. Can I get a refund?

- The LLM does not know your hospital's policies.
- So instead of relying on memory, it searches the hospital documents first.
- This is exactly where Retrieval Augmented Generation (RAG) comes in.

INTRODUCTION TO RAG:
Retrieval Augmented Generation (RAG)
large language models are trained on massive datasets.
however:
- they do not know our company's internal documetns.
- they cannot access newly updated information.
- sometimes they generate incorrect answers confidently.

instead of asking the LLM to remember everything, we first retrieve the relevant information from our documents and provide it to the LLM.This technique is called Retrieval Augmented Generation (RAG).

##### What is RAG ?
###### Retrieval Augmented Generation (RAG) is a technique where an LLM first searches knowledge base for relevant information before generating an answer.

###### instead of relying only on what it learned during training, it also uses external documents.

user question -> retreiver -> relevant documents -> LLM -> Final Answer

##### Why LLMs Hallucinate ?
###### LLMs predict the next word based on the probability. They do not verify facts before answering. If they don't know something, they may still generate an answer that sounds correct but is actually wrong. This behaviour is called hallucination.

example - 
user: does our hospital allow pets?
llm: yes, pets are allowed.
reality: the hospital policy may actually prohibit pets.


##### RAG Architecture

user question -> embedding model -> retriever -> knowledge base -> relevant documents -> prompt augmentation -> llm -> answer

##### Documents vs Embeddings
###### documents are plain text files that contain information.
###### ex - refund policy, appointment rules, insurance policy.

###### every document is converted into numerical representation called embedding. Embeddings are vectors (list of numbers) that capture the semantic meaning of text. Documents with similar meanings have similar embeddings. Retriever compares embeddings instead of comparing raw text.

##### Retrieval process

user asks a question.
question is converted into an embedding.
retriever searches the knowledge base.
most relevant documents are retrieved.
retrieved documents are sent to the llm.
llm generates the final answer.

##### Prompt Augmentation
instead of asking "Can I cancel my appointment", we ask "context (appointment policy) + Can I cancel my appointment?"

In [ ]:
# Airline Policy Knowledge Base (Hospital Knowledge Base)

In [1]:
hospital_policies = [
    "Appointments can be cancelled up to 24 hours before the scheduled time.",
    "Refunds are processed within 7 working days.",
    "Visiting hours are from 10 AM to 6 PM.",
    "Emergency services are available 24/7.",
    "Dental treatmetns are not covered under basic insurance.",
    "Patients mist carry a valid ID during resgitration.",
    "One attendent is allowed per patient.",
    "Medical reports can be collected after 3 working days."
]
print("Knowledge Base Loaded!!!")

Knowledge Base Loaded!!!


In [2]:
def retrieve_documents(question):
    results = []

    question = question.lower()
    for doc in hospital_policies:
        if any(word in doc.lower() for word in question.split()):
            results.append(doc)
    
    return results

In [3]:
## test retriever
question = "Can I cancel my appointment?"
documents = retrieve_documents(question)
print(documents)

['Appointments can be cancelled up to 24 hours before the scheduled time.', 'Refunds are processed within 7 working days.', 'Visiting hours are from 10 AM to 6 PM.', 'Emergency services are available 24/7.', 'Dental treatmetns are not covered under basic insurance.', 'Patients mist carry a valid ID during resgitration.', 'One attendent is allowed per patient.', 'Medical reports can be collected after 3 working days.']


In [4]:
## prompt augmentation
question = "Can I cancel my appointment?"
documents = retrieve_documents(question)
context = "\n".join(documents)

prompt = f""" 
Context: 

{context}

Question:

{question}

Answer only using the context.
"""

print(prompt)

 
Context: 

Appointments can be cancelled up to 24 hours before the scheduled time.
Refunds are processed within 7 working days.
Visiting hours are from 10 AM to 6 PM.
Emergency services are available 24/7.
Dental treatmetns are not covered under basic insurance.
Patients mist carry a valid ID during resgitration.
One attendent is allowed per patient.
Medical reports can be collected after 3 working days.

Question:

Can I cancel my appointment?

Answer only using the context.



the above prompt consits not only the question, but: 
- retrieved context
- user question
- instructions

In [5]:
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()

# -----------------------------
# Knowledge Base
# -----------------------------
hospital_policies = [
    "Appointments can be cancelled up to 24 hours before the scheduled time.",
    "Refunds are processed within 7 working days.",
    "Visiting hours are from 10 AM to 6 PM.",
    "Emergency services are available 24/7.",
    "Dental treatments are not covered under basic insurance.",
    "Patients must carry a valid ID during registration.",
    "One attendant is allowed per patient.",
    "Medical reports can be collected after 3 working days."
]

# -----------------------------
# Retriever
# -----------------------------
def retrieve_documents(question):
    results = []

    question = question.lower()

    for document in hospital_policies:
        if any(word in document.lower() for word in question.split()):
            results.append(document)

    return results

# -----------------------------
# LLM
# -----------------------------
def hospital_llm(prompt):

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a Hospital AI Assistant.\n"
                    "Answer ONLY using the provided context.\n"
                    "If the answer is not present in the context, "
                    "reply with: 'I couldn't find that information in the hospital policy.'"
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

# -----------------------------
# User Question
# -----------------------------
question = input("Ask your question: ")

# -----------------------------
# Retrieval
# -----------------------------
documents = retrieve_documents(question)

context = "\n".join(documents)

# -----------------------------
# Prompt Augmentation
# -----------------------------
prompt = f"""
Context:
{context}

Question:
{question}

Answer only using the context.
"""

# (Optional) Display the prompt to understand Prompt Augmentation
print("\n===== Prompt Sent to LLM =====")
print(prompt)

# -----------------------------
# Generate Answer
# -----------------------------
response = hospital_llm(prompt)

print("\n===== Assistant =====")
print(response)


===== Prompt Sent to LLM =====

Context:


Question:


Answer only using the context.


===== Assistant =====
I couldn't find that information in the hospital policy.


In [ ]:
How LLM Tool Calling Really Works
• What is Tool Calling?
• LLM vs Python Functions
• Why LLMs cannot execute code directly
• Tool Definitions
• JSON Schema
• Function Name
• Parameters
• Required Fields
• Tool Choice
• Tool Arguments
• Assistant Tool Calls
• Tool Messages
• Final Response Generation